In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
import lightgbm as lgb

In [2]:
# ======================
# 1. Load Data 
# ======================

train_data = pd.read_csv('analysis_data.csv')
scoring_data = pd.read_csv('scoring_data.csv')
TARGET_RAW = 'monthly_spend'
TARGET_CLEAN = 'monthly_spend_cleaned'
ID_COL = 'customer_id'

In [3]:
# ======================
# 2. Basic Data Cleaning
# ======================
# Only cap the target variable to avoid data leakage

upper_limit = train_data[TARGET_RAW].quantile(0.99)
train_data[TARGET_CLEAN] = train_data[TARGET_RAW].clip(upper=upper_limit)

In [ ]:
# ======================
# 3. Feature Engineering (Sub4 ONLY changes here)
# ======================

def create_features(df):
    df = df.copy()
    
    # Base features
    df['avg_limit_per_card'] = df['credit_limit'] / df['num_credit_cards'].replace(0, 1)
    df['estimated_monthly_volume'] = df['num_transactions'] * df['avg_transaction_value']
    df['monthly_income'] = df['annual_income'] / 12
    df['spend_to_income_ratio'] = df['avg_transaction_value'] / df['monthly_income'].replace(0, 1)
    
    # Sub4 features
    df['credit_utilization'] = df['avg_transaction_value'] / df['credit_limit'].replace(0, 1)
    df['debt_to_income_ratio'] = df['num_credit_cards'] / df['annual_income'].replace(0, 1)
    df['transactions_per_card'] = df['num_transactions'] / df['num_credit_cards'].replace(0, 1)
    df['credit_score_tier'] = pd.cut(df['credit_score'], bins=5, labels=False)
    
    # Sub5 features
    df['age_life_stage'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 55, 100], labels=False)
    df['credit_score_fico_bin'] = pd.cut(df['credit_score'], bins=[0, 580, 670, 740, 800, 850], labels=False)
    df['annual_income_quintile'] = pd.qcut(df['annual_income'], q=5, labels=False, duplicates='drop')
    
    return df

In [ ]:
# ======================
# 4. Define Feature Lists
# ======================

def get_feature_lists():
    base_numerical = [
        'age', 'annual_income', 'credit_score', 'num_credit_cards', 
        'num_transactions', 'avg_transaction_value', 'credit_limit',
        'avg_limit_per_card', 'estimated_monthly_volume', 'spend_to_income_ratio'
    ]
    
    sub4_numerical = [
        'credit_utilization', 'debt_to_income_ratio', 'transactions_per_card', 'credit_score_tier'
    ]
    
    sub5_numerical = [
        'age_life_stage', 'credit_score_fico_bin', 'annual_income_quintile'
    ]
    
    numerical_features = base_numerical + sub4_numerical + sub5_numerical
    categorical_features = ['education_level', 'gender', 'region', 'employment_status', 'card_type']
    
    return numerical_features, categorical_features

In [ ]:
# ======================
# 5. Model Training with 5-Fold Cross Validation
# ======================

def train_and_predict(X, y, X_scoring):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []
    test_predictions = np.zeros(len(X_scoring))
    
    print("=" * 60)
    print("Starting 5-Fold Cross Validation (Sub6: Random Forest)")
    print("=" * 60)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # ----------------------
        # Sub6 NEW MODEL (Single variable change)
        # ----------------------
        model = RandomForestRegressor(
            n_estimators=200,    # Number of trees (more = better, but slower)
            max_depth=12,         # Prevent overfitting
            min_samples_split=10, # Prevent overfitting
            n_jobs=-1,            # Use all CPU cores
            random_state=42
        )
        
        model.fit(X_train, y_train)
        
        # Evaluate on validation set
        val_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        fold_scores.append(rmse)
        print(f"Fold {fold} RMSE: {rmse:.4f}")
        
        # Predict on test set
        test_pred = model.predict(X_scoring)
        test_predictions += test_pred / 5
    
    # Print results
    print("\n" + "=" * 60)
    print(f"Sub6 Cross Validation Results:")
    print(f"  Previous Mean CV RMSE (Sub5): 243.6979")
    print(f"  Individual Fold RMSE: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"  Mean CV RMSE: {np.mean(fold_scores):.4f}")
    print(f"  Standard Deviation: {np.std(fold_scores):.4f}")
    print(f"  Improvement: {243.6979 - np.mean(fold_scores):.4f}")
    print("=" * 60)
    
    return test_predictions

In [ ]:
# ======================
# 6. Main Pipeline
# ======================

def main():
    # 特征工程
    train_fe = create_features(train_data)
    scoring_fe = create_features(scoring_data)
    
    # 获取特征列
    num_features, cat_features = get_feature_lists(train_fe)
    all_features = num_features + cat_features
    
    # 分离特征和目标
    X = train_fe[all_features]
    y = train_fe[TARGET_CLEAN]
    X_scoring = scoring_fe[all_features]
    
    # 类别特征编码（独热编码，共用）
    X = pd.get_dummies(X, columns=cat_features, drop_first=True)
    X_scoring = pd.get_dummies(X_scoring, columns=cat_features, drop_first=True)
    X, X_scoring = X.align(X_scoring, join='left', axis=1, fill_value=0)
    
    # 缺失值填充（中位数，共用）
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    X_scoring_imputed = pd.DataFrame(imputer.transform(X_scoring), columns=X_scoring.columns)
    
    # 训练模型并预测
    print('='*50)
    print('开始训练...')
    print('='*50)
    predictions = train_and_predict(X_imputed, y, X_scoring_imputed)
    
    # 非负处理（共用）
    predictions = np.maximum(predictions, 0)
    
    # 生成提交文件
    submission = pd.DataFrame({
        ID_COL: scoring_data[ID_COL],
        TARGET_RAW: predictions
    })
    submission.to_csv('submission_v6.csv', index=False)  # 每次改一下版本号
    print('\n提交文件已生成！')

if __name__ == '__main__':
    main()

开始训练...
Fold 1 RMSE: 220.2957
Fold 2 RMSE: 226.4161
Fold 3 RMSE: 226.0194
Fold 4 RMSE: 220.2457
Fold 5 RMSE: 224.4684

Mean CV RMSE: 223.4891 (±2.7073)

提交文件已生成！
